# RappiPlus Analytics: From Data to Business Decisions

## Overview

This project evaluates the performance of the **RappiPlus** subscription service using end-to-end data analysis — from data quality validation through statistical testing — to surface actionable business insights.

### Datasets

| Dataset | Description |
|---|---|
| `rappiplus_orders_raw.csv` | Order-level data: quantities, unit prices, discounts, and revenue |
| `rappiplus_catalog.csv` | Product catalog: unit costs, categories, and suppliers |
| `rappiplus_marketing_spend.csv` | Marketing investment by channel and country |
| `events / users / user_activity` (SQL) | User behavior and activity data |
| `experiment_checkout_ui.csv` | A/B test results for checkout UI redesign |

### Analysis Pipeline

1. 🔍 **Data Quality** — Validate and clean all datasets before analysis
2. 💰 **Profitability Analysis** — Calculate revenue, costs, and profit margins
3. 🛒 **Conversion Funnel** — Identify drop-off points across the user journey
4. 🔁 **Cohort Retention** — Measure weekly user retention by registration cohort
5. 🧪 **A/B Testing** — Statistically validate the impact of the checkout UI change
6. 📊 **BI Dashboard** — Communicate findings through an executive dashboard


---

## 🔹 Step 1: Data Loading & Quality Validation

### 1.1 Load Datasets

Load the three core CSV files and perform an initial structural inspection.

---

In [ ]:
import pandas as pd

# Load datasets
orders   = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog  = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [ ]:
# Initial inspection — orders
orders.info()
print()
orders.head(10)

In [ ]:
# Initial inspection — catalog
catalog.info()
print()
catalog.head(10)

In [ ]:
# Initial inspection — marketing
marketing.info()
print()
marketing.head(10)

---

### 1.2 Data Cleaning

The cleaning process covers all three datasets and addresses:
- Date parsing and format standardization
- Removal of records with invalid numeric values (negatives, zeros)
- Amount consistency checks
- Duplicate detection and removal
- Categorical variable validation

---

In [ ]:
# ── ORDERS: Data Cleaning ─────────────────────────────────────────────────

# Parse datetime column
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')
orders.info()

# Filter out records with non-positive quantity or total amount
orders_clean = orders[
    (orders['cantidad'] > 0) &
    (orders['monto_total'] > 0)
].copy()

invalid_records = orders[~((orders['cantidad'] > 0) & (orders['monto_total'] > 0))]

print(f"Invalid records removed  : {len(invalid_records)}")
print(f"  quantity <= 0          : {(orders['cantidad'] <= 0).sum()}")
print(f"  quantity NaN           : {orders['cantidad'].isna().sum()}")
print(f"  total_amount <= 0      : {(orders['monto_total'] <= 0).sum()}")
print(f"  total_amount NaN       : {orders['monto_total'].isna().sum()}")
print(f"Remaining records        : {len(orders_clean)}")


In [ ]:
# Amount consistency check — computed vs recorded total (2-cent tolerance)
orders_clean['monto_calculado'] = (
    orders_clean['cantidad'] * orders_clean['precio_unitario']
) - orders_clean['monto_descuento']

orders_clean['diferencia'] = orders_clean['monto_total'] - orders_clean['monto_calculado']
inconsistencias = orders_clean[orders_clean['diferencia'].abs() > 0.02]

print(f"Amount inconsistencies found: {len(inconsistencias)}")


In [ ]:
# Duplicate detection and removal
exact_dupes     = orders_clean.duplicated().sum()
order_id_dupes  = orders_clean.duplicated(subset=['id_pedido']).sum()

print(f"Exact duplicates           : {exact_dupes}")
print(f"Duplicates by order ID     : {order_id_dupes}")

if order_id_dupes > 0:
    duped = orders_clean[orders_clean.duplicated(subset=['id_pedido'], keep=False)]
    print("\nSample duplicated orders:")
    print(duped[['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'monto_total']].head(10))

    dupes_count = duped.groupby('id_pedido').size()
    print(f"\nOrders with 2 records  : {(dupes_count == 2).sum()}")
    print(f"Orders with 3+ records : {(dupes_count > 2).sum()}")

print(f"\nBefore deduplication : {len(orders_clean)} records")
orders_clean_no_duplicates = orders_clean.drop_duplicates(subset=['id_pedido'], keep='first')
print(f"After deduplication  : {len(orders_clean_no_duplicates)} records")
print(f"Removed              : {len(orders_clean) - len(orders_clean_no_duplicates)} records")


In [ ]:
# Categorical variable review — null patterns and consistency
categorical_cols = orders_clean_no_duplicates.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {categorical_cols}")

for col in categorical_cols:
    nulls = orders_clean_no_duplicates[col].isnull().sum()
    print(f"  {col:<30} nulls: {nulls}")

    # Check for leading/trailing whitespace
    spaces = orders_clean_no_duplicates[col].str.contains(r'^\s+|\s+$', na=False).sum()
    if spaces > 0:
        print(f"    ⚠  {spaces} values with extra whitespace")

    # Check for case inconsistencies
    unique_vals = orders_clean_no_duplicates[col].dropna().unique()
    for i, v1 in enumerate(unique_vals):
        for v2 in unique_vals[i+1:]:
            if isinstance(v1, str) and isinstance(v2, str):
                if v1.lower() == v2.lower() and v1 != v2:
                    print(f"    ⚠  Case mismatch: '{v1}' vs '{v2}'")


In [ ]:
# Standardize country name casing and finalize orders dataset
orders_final = orders_clean_no_duplicates.copy()
orders_final['pais'] = orders_final['pais'].str.title()

# Null impact assessment
multi_null_mask = (
    orders_final['fuente_referencia'].isnull() &
    orders_final['nombre_producto'].isnull() &
    orders_final['categoria_producto'].isnull()
)
multi_null_records = orders_final[multi_null_mask]
revenue_share = (multi_null_records['monto_total'].sum() / orders_final['monto_total'].sum()) * 100

print(f"Records with multiple nulls : {len(multi_null_records)}")
print(f"Revenue share              : {revenue_share:.2f}%  — retained for country-level analysis")
print(f"\nFinal orders dataset       : {len(orders_final)} records")


In [ ]:
# ── CATALOG: Data Cleaning ────────────────────────────────────────────────

# Filter invalid unit costs
catalog_clean = catalog[catalog['costo_unitario'] > 0].copy()
print(f"Invalid cost records removed : {len(catalog) - len(catalog_clean)}")

# Duplicate check
print(f"Exact duplicates             : {catalog_clean.duplicated().sum()}")
print(f"Duplicates by product name   : {catalog_clean.duplicated(subset=['nombre_producto']).sum()}")

catalog_final = catalog_clean.drop_duplicates(subset=['nombre_producto'], keep='first')

print(f"\nFinal catalog dataset:")
print(f"  Records    : {len(catalog_final)}")
print(f"  Products   : {catalog_final['nombre_producto'].nunique()}")
print(f"  Categories : {catalog_final['categoria_producto'].nunique()}")
print(f"  Suppliers  : {catalog_final['proveedor'].nunique()}")

# Category distribution
print("\nCategory distribution:")
for val, cnt in catalog_final['categoria_producto'].value_counts().items():
    print(f"  {val}: {cnt} ({cnt/len(catalog_final):.1%})")


In [ ]:
# ── MARKETING: Data Cleaning ──────────────────────────────────────────────

# Parse date column
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors='coerce')
marketing.info()

# Filter non-positive spend records
marketing_clean = marketing[marketing['gasto'] > 0].copy()
print(f"\nInvalid spend records removed : {len(marketing) - len(marketing_clean)}")

# Duplicate check by composite key: date + country + channel
dupes = marketing_clean.duplicated(subset=['fecha', 'pais', 'canal']).sum()
print(f"Duplicates (date-country-channel) : {dupes}")

if dupes > 0:
    sample = marketing_clean[marketing_clean.duplicated(subset=['fecha', 'pais', 'canal'], keep=False)]
    print(sample[['fecha', 'pais', 'canal', 'gasto']].head(10))

marketing_final = marketing_clean.drop_duplicates(subset=['fecha', 'pais', 'canal'], keep='first')

print(f"\nFinal marketing dataset:")
print(f"  Records         : {len(marketing_final)}")
print(f"  Countries       : {marketing_final['pais'].nunique()}")
print(f"  Channels        : {marketing_final['canal'].nunique()}")
print(f"  Date range      : {marketing_final['fecha'].min().date()} → {marketing_final['fecha'].max().date()}")
print(f"  Total spend     : ${marketing_final['gasto'].sum():,.2f}")


---
**Export:** Cleaned datasets are saved to CSV for use in the BI dashboard (Step 6).

In [ ]:
# Export cleaned datasets
orders_final.to_csv('orders_clean.csv', index=False)
catalog_final.to_csv('catalog_clean.csv', index=False)
marketing_final.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Step 2: Profitability Analysis

### 2.1 Key Business KPIs

Using the three cleaned datasets, this section calculates the core financial metrics:

**Part 1 — Business Profitability**
- Total revenue
- Total product cost (from catalog unit costs)
- Total marketing investment
- Gross and net profit, and corresponding margins

**Part 2 — Sales Behavior**
- Average order value (AOV)
- Average units per order
- Top-selling products
- Marketing spend breakdown by channel and country


In [ ]:
# ── Part 1: Business Profitability ────────────────────────────────────────

# Total revenue
total_revenue = orders_final['monto_total'].sum()
print(f"Total Revenue           : ${total_revenue:,.2f}")

# Total product cost — join orders with catalog to get unit costs
orders_with_costs = orders_final.merge(
    catalog_final[['nombre_producto', 'costo_unitario']],
    on='nombre_producto',
    how='left'
)
orders_with_costs['product_total_cost'] = (
    orders_with_costs['cantidad'] * orders_with_costs['costo_unitario']
)
total_product_cost = orders_with_costs['product_total_cost'].sum()
print(f"Total Product Cost      : ${total_product_cost:,.2f}")

# Total marketing spend
total_marketing = marketing_final['gasto'].sum()
print(f"Total Marketing Spend   : ${total_marketing:,.2f}")

# Profit metrics
gross_profit = total_revenue - total_product_cost
net_profit   = gross_profit - total_marketing

print(f"\nGross Profit            : ${gross_profit:,.2f}  ({gross_profit/total_revenue:.1%} margin)")
print(f"Net Profit              : ${net_profit:,.2f}   ({net_profit/total_revenue:.1%} margin)")
print(f"\nBusiness is profitable  : {'✅ Yes' if net_profit > 0 else '❌ No'}")


In [ ]:
# ── Part 2: Sales Behavior ─────────────────────────────────────────────────

# Average order value
aov = orders_final['monto_total'].mean()
print(f"Average Order Value (AOV)       : ${aov:.2f}")

# Average units per order
avg_units = orders_final['cantidad'].mean()
print(f"Average Units per Order         : {avg_units:.2f}")

# Top-selling products by units sold
top_products = (
    orders_final.groupby('nombre_producto')['cantidad']
    .sum()
    .sort_values(ascending=False)
)
print(f"\nTop 5 Products by Units Sold:")
for i, (product, qty) in enumerate(top_products.head(5).items(), 1):
    print(f"  {i}. {product}: {qty:.0f} units")

# Marketing spend by channel
spend_by_channel = (
    marketing_final.groupby('canal')['gasto']
    .sum()
    .sort_values(ascending=False)
)
print(f"\nMarketing Spend by Channel:")
for channel, spend in spend_by_channel.items():
    print(f"  {channel}: ${spend:,.2f}")

# Marketing spend by country
spend_by_country = (
    marketing_final.groupby('pais')['gasto']
    .sum()
    .sort_values(ascending=False)
)
print(f"\nMarketing Spend by Country:")
for country, spend in spend_by_country.items():
    print(f"  {country}: ${spend:,.2f}")


---

## 🔹 Step 3: Conversion Funnel Analysis

This section analyzes user behavior across the purchase journey using event-level data from the SQL database.

**Funnel stages (in order):**
`first_visit` → `add_to_cart` → `select_item` → `begin_checkout` → `add_payment_info` → `purchase`

The analysis answers:
- How many unique users reach each stage?
- What is the step-to-step conversion rate?
- Where does the largest drop-off occur?
- What is the end-to-end conversion rate?

---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# Database connection
db_config = {
    'user': 'practicum_student',
    'pwd' : 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db'  : 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{user}:{pwd}@{host}:{port}/{db}'.format(**db_config)
engine = create_engine(connection_string, connect_args={'sslmode': 'require'})

In [ ]:
# Explore events table
events = pd.read_sql('SELECT * FROM events;', con=engine)
events.head()

In [ ]:
# Part 1 — Unique users per funnel stage
query_funnel_totals = """
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS unique_users
FROM events
GROUP BY nombre_evento
ORDER BY
    CASE nombre_evento
        WHEN 'first_visit'       THEN 1
        WHEN 'add_to_cart'       THEN 2
        WHEN 'select_item'       THEN 3
        WHEN 'begin_checkout'    THEN 4
        WHEN 'add_payment_info'  THEN 5
        WHEN 'purchase'          THEN 6
        ELSE 7
    END;
"""

funnel_totals = pd.read_sql(query_funnel_totals, con=engine)
funnel_totals

In [ ]:
# Part 2 — Step-to-step conversion rates
query_conversion = """
WITH funnel_ordered AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS unique_users,
        CASE nombre_evento
            WHEN 'first_visit'       THEN 1
            WHEN 'add_to_cart'       THEN 2
            WHEN 'select_item'       THEN 3
            WHEN 'begin_checkout'    THEN 4
            WHEN 'add_payment_info'  THEN 5
            WHEN 'purchase'          THEN 6
            ELSE 7
        END AS stage_order
    FROM events
    GROUP BY nombre_evento
),
funnel_with_lag AS (
    SELECT
        nombre_evento,
        unique_users,
        stage_order,
        LAG(unique_users) OVER (ORDER BY stage_order) AS prev_stage_users
    FROM funnel_ordered
)
SELECT
    nombre_evento,
    unique_users,
    prev_stage_users,
    CASE
        WHEN prev_stage_users IS NOT NULL
        THEN ROUND((unique_users::DECIMAL / prev_stage_users) * 100, 2)
        ELSE 100.0
    END AS conversion_rate_pct,
    CASE
        WHEN prev_stage_users IS NOT NULL
        THEN prev_stage_users - unique_users
        ELSE 0
    END AS users_lost
FROM funnel_with_lag
ORDER BY stage_order;
"""

conversion = pd.read_sql(query_conversion, con=engine)
conversion

---

In [ ]:
# End-to-end conversion rate: first visit → purchase
query_overall_conversion = """
WITH totals AS (
    SELECT
        SUM(CASE WHEN nombre_evento = 'first_visit' THEN 1 ELSE 0 END) AS initial_visitors,
        SUM(CASE WHEN nombre_evento = 'purchase'    THEN 1 ELSE 0 END) AS purchasers
    FROM (
        SELECT DISTINCT id_usuario, nombre_evento
        FROM events
        WHERE nombre_evento IN ('first_visit', 'purchase')
    ) t
)
SELECT
    initial_visitors,
    purchasers,
    ROUND((purchasers::DECIMAL / initial_visitors) * 100, 2) AS overall_conversion_pct
FROM totals;
"""

overall_conversion = pd.read_sql(query_overall_conversion, con=engine)
overall_conversion

## 🔹 Step 4: Cohort Retention Analysis

This section measures whether users return after registering, using a weekly retention model broken down by monthly registration cohort.

**Approach:**
1. Group users into cohorts based on their **month of registration**.
2. Calculate weekly retention: how many users remain active in each of the first three weeks post-registration.
3. Express retention as a percentage of each cohort's initial size.

**Tables used:** `users`, `user_activity`


In [ ]:
# Explore users table
users = pd.read_sql('SELECT * FROM users;', con=engine)
users.head(3)

In [ ]:
# Explore user_activity table
user_activity = pd.read_sql('SELECT * FROM user_activity;', con=engine)
user_activity.head(3)

In [ ]:
# Step 1 — Cohort size by registration month
query_cohorts = """
WITH user_cohorts AS (
    SELECT
        id_usuario,
        CAST(fecha_registro AS DATE) AS registration_date,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohort_month
    FROM users
)
SELECT
    cohort_month,
    COUNT(DISTINCT id_usuario) AS registered_users
FROM user_cohorts
GROUP BY cohort_month
ORDER BY cohort_month;
"""

cohorts = pd.read_sql(query_cohorts, con=engine)
cohorts

In [ ]:
# Step 2 — Weekly retention by cohort (raw counts)
query_weekly_retention = """
WITH user_cohorts AS (
    SELECT
        id_usuario,
        CAST(fecha_registro AS DATE) AS registration_date,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohort_month
    FROM users
),
weekly_activity AS (
    SELECT
        uc.cohort_month,
        uc.id_usuario,
        ua.dias_despues_registro,
        ua.activo,
        CASE
            WHEN ua.dias_despues_registro BETWEEN  1 AND  7 THEN 'week_1'
            WHEN ua.dias_despues_registro BETWEEN  8 AND 14 THEN 'week_2'
            WHEN ua.dias_despues_registro BETWEEN 15 AND 21 THEN 'week_3'
            ELSE 'other'
        END AS week_period
    FROM user_cohorts uc
    JOIN user_activity ua ON uc.id_usuario = ua.id_usuario
    WHERE ua.dias_despues_registro <= 21
),
cohort_sizes AS (
    SELECT cohort_month, COUNT(DISTINCT id_usuario) AS initial_users
    FROM user_cohorts
    GROUP BY cohort_month
),
retained_by_week AS (
    SELECT
        cohort_month,
        week_period,
        COUNT(DISTINCT CASE WHEN activo = 1 THEN id_usuario END) AS retained_users
    FROM weekly_activity
    WHERE week_period IN ('week_1', 'week_2', 'week_3')
    GROUP BY cohort_month, week_period
)
SELECT
    cs.cohort_month,
    cs.initial_users,
    COALESCE(MAX(CASE WHEN rw.week_period = 'week_1' THEN rw.retained_users END), 0) AS retained_w1,
    COALESCE(MAX(CASE WHEN rw.week_period = 'week_2' THEN rw.retained_users END), 0) AS retained_w2,
    COALESCE(MAX(CASE WHEN rw.week_period = 'week_3' THEN rw.retained_users END), 0) AS retained_w3
FROM cohort_sizes cs
LEFT JOIN retained_by_week rw ON cs.cohort_month = rw.cohort_month
GROUP BY cs.cohort_month, cs.initial_users
ORDER BY cs.cohort_month;
"""

weekly_retention = pd.read_sql(query_weekly_retention, con=engine)
weekly_retention

In [ ]:
# Step 3 — Cohort retention table (percentage format)
query_cohort_retention = """
WITH user_cohorts AS (
    SELECT
        id_usuario,
        CAST(fecha_registro AS DATE) AS registration_date,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohort_month
    FROM users
),
cohort_sizes AS (
    SELECT cohort_month, COUNT(DISTINCT id_usuario) AS initial_users
    FROM user_cohorts
    GROUP BY cohort_month
),
weekly_activity AS (
    SELECT
        uc.cohort_month,
        uc.id_usuario,
        ua.dias_despues_registro,
        ua.activo,
        CASE
            WHEN ua.dias_despues_registro BETWEEN  1 AND  7 THEN 'week_1'
            WHEN ua.dias_despues_registro BETWEEN  8 AND 14 THEN 'week_2'
            WHEN ua.dias_despues_registro BETWEEN 15 AND 21 THEN 'week_3'
        END AS week_period
    FROM user_cohorts uc
    JOIN user_activity ua ON uc.id_usuario = ua.id_usuario
    WHERE ua.dias_despues_registro <= 21 AND ua.activo = 1
),
retained_by_week AS (
    SELECT
        cohort_month,
        week_period,
        COUNT(DISTINCT id_usuario) AS retained_users
    FROM weekly_activity
    WHERE week_period IN ('week_1', 'week_2', 'week_3')
    GROUP BY cohort_month, week_period
)
SELECT
    cs.cohort_month,
    cs.initial_users,
    ROUND(
        COALESCE(MAX(CASE WHEN rw.week_period = 'week_1' THEN rw.retained_users END), 0)
        * 100.0 / cs.initial_users, 2
    ) AS week_1_pct,
    ROUND(
        COALESCE(MAX(CASE WHEN rw.week_period = 'week_2' THEN rw.retained_users END), 0)
        * 100.0 / cs.initial_users, 2
    ) AS week_2_pct,
    ROUND(
        COALESCE(MAX(CASE WHEN rw.week_period = 'week_3' THEN rw.retained_users END), 0)
        * 100.0 / cs.initial_users, 2
    ) AS week_3_pct
FROM cohort_sizes cs
LEFT JOIN retained_by_week rw ON cs.cohort_month = rw.cohort_month
GROUP BY cs.cohort_month, cs.initial_users
ORDER BY cs.cohort_month;
"""

cohort_retention = pd.read_sql(query_cohort_retention, con=engine)
cohort_retention

---

## 🔹 Step 5: A/B Test — Checkout UI Impact

This section evaluates whether a redesigned checkout UI significantly improves purchase conversion rates.

**Dataset:** `experiment_checkout_ui.csv`  
**Key metric:** `convirtio` — binary flag (1 = purchase completed, 0 = no purchase)

**Hypotheses:**
- **H₀ (Null):** The conversion rate is the same for both variants. Any observed difference is due to random sampling variation.
- **H₁ (Alternative):** The conversion rate differs between variants. The UI change has a statistically significant effect on purchase behavior.

**Statistical test:** Two-proportion Z-test (confirmed by Chi-square for robustness)  
**Significance level:** α = 0.05

---

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest

# Load experiment data
df = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')

alpha = 0.05
print(f"Significance level (α) : {alpha}")
print(f"Confidence level       : {(1-alpha)*100:.0f}%")
print(f"Total sample size      : {len(df):,} users")


In [ ]:
# Conversion rates by variant
conversions = df.groupby('variante')['convirtio'].sum()
totals       = df.groupby('variante')['convirtio'].count()

rate_control   = conversions['control']    / totals['control']
rate_treatment = conversions['tratamiento'] / totals['tratamiento']

print("Conversion Summary:")
print(f"  Control   : {conversions['control']:,} / {totals['control']:,}  → {rate_control:.2%}")
print(f"  Treatment : {conversions['tratamiento']:,} / {totals['tratamiento']:,}  → {rate_treatment:.2%}")
print(f"  Absolute difference : {abs(rate_treatment - rate_control):.2%}")

# Two-proportion Z-test
successes    = [conversions['control'], conversions['tratamiento']]
observations = [totals['control'], totals['tratamiento']]
z_stat, p_value_z = proportions_ztest(successes, observations)

print(f"\nTwo-Proportion Z-Test:")
print(f"  z-statistic : {z_stat:.4f}")
print(f"  p-value     : {p_value_z:.6f}")

if p_value_z < alpha:
    better = 'Treatment' if rate_treatment > rate_control else 'Control'
    print(f"  → Reject H₀: Significant difference detected (p < {alpha})")
    print(f"     The {better} variant yields a higher conversion rate.")
else:
    print(f"  → Fail to reject H₀: No significant difference found (p ≥ {alpha})")
    print(f"     Conversion rates are statistically equivalent.")


In [ ]:
# Chi-square test — cross-validation
contingency = pd.crosstab(df['variante'], df['convirtio'])
print("Contingency Table:")
print(contingency)

chi2_stat, p_value_chi2, dof, expected = chi2_contingency(contingency)

print(f"\nChi-Square Test:")
print(f"  χ² statistic      : {chi2_stat:.4f}")
print(f"  p-value           : {p_value_chi2:.6f}")
print(f"  Degrees of freedom: {dof}")
print(f"  Min expected freq : {expected.min():.2f}  (all ≥ 5: {'Yes' if expected.min() >= 5 else 'No'})")

# Consistency check — z² ≈ χ² for 2×2 tables
print(f"\nMethod Consistency:")
print(f"  z² = {z_stat**2:.4f}   χ² = {chi2_stat:.4f}   Δ = {abs(z_stat**2 - chi2_stat):.6f}")
consistent = (p_value_z < alpha) == (p_value_chi2 < alpha)
print(f"  Both tests agree  : {'Yes' if consistent else 'No'}")


In [ ]:
# Final interpretation and recommendation
print("A/B Test — Final Results")
print("=" * 50)
print(f"  Sample size : {len(df):,} users")
print(f"  Control     : {totals['control']:,} users  ({rate_control:.2%} conversion)")
print(f"  Treatment   : {totals['tratamiento']:,} users  ({rate_treatment:.2%} conversion)")
print(f"  α = {alpha}   |   z = {z_stat:.4f}   |   p = {p_value_z:.6f}")
print()

if p_value_z < alpha:
    better = 'Treatment' if rate_treatment > rate_control else 'Control'
    print(f"Conclusion: Statistically significant difference detected.")
    print(f"  The {better} variant outperforms by {abs(rate_treatment - rate_control):.2%}.")
    print(f"  Recommendation: Deploy the {better} variant.")
else:
    print("Conclusion: No statistically significant difference detected.")
    print("  Recommendation: Extend the experiment or maintain the current version.")


---

## 🔹 Step 6: BI Dashboard

The final step translates the analysis into an interactive executive dashboard built in Power BI or Tableau.

**Input files:** `orders_clean.csv`, `catalog_clean.csv`, `marketing_clean.csv`

### Data Model
- `orders.nombre_producto` → `catalog.nombre_producto`
- `orders.fecha_pedido` → date dimension table (for YTD / YoY comparisons)

---

### Dashboard 1 — Executive Overview

**KPI Cards:** Total Revenue · Net Profit · Marketing Spend · AOV · Avg Units/Order  
**Visuals:**
- Line chart: monthly revenue / profit trend
- Line chart: YTD cumulative revenue
- Bar chart: revenue and profit by product / category

---

### Dashboard 2 — Detail / Drill-through

**Purpose:** Allow exploration from high-level KPIs down to individual orders.

**Visuals:**
- Order detail table: product · quantity · revenue · cost · profit (conditional color formatting)
- Bar chart: units sold per product
- Drill-through: click a product → see all related orders
- Slicers: date range · product category


---

## 🚀 Dashboard Link

The interactive dashboard is published and accessible at the link below.


In [ ]:
# Dashboard links
# Power BI Service / Tableau Public: <insert link>
# Project files (OneDrive / Google Drive): <insert link>
